# 23 — Validação da ingestão local com LiteParse

Esta etapa aproxima o Fato ou Fake de uma experiência semelhante ao NotebookLM: o documento enviado é transformado em texto estruturado antes da extração de alegações e da busca de evidências. O objetivo aqui é validar apenas a fundação de ingestão, sem produzir um veredito científico.

## Critérios de aceite

1. O parser funciona localmente, sem enviar o PDF a uma nuvem.
2. O resultado contém texto, número de páginas e nome do parser.
3. PDFs com texto insuficiente são rejeitados explicitamente.
4. O texto extraído pode ser entregue ao estágio de extração de alegações.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'src'))

import liteparse

from fatofake.document_parsing import LiteParseDocumentParser

print('LiteParse:', liteparse.__version__)

LiteParse: 2.15.0


## Documento controlado

Geramos um PDF pequeno e conhecido para que o teste seja reproduzível e não dependa da rede.

In [2]:
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt

sample_text = (
    'Controlled scientific article. This document evaluates whether a treatment '
    'reduces symptoms in adults. The reported result is illustrative and must not '
    'be interpreted as clinical evidence. The purpose is validating local parsing.'
)

with TemporaryDirectory() as temporary_dir:
    pdf_path = Path(temporary_dir) / 'controlled_article.pdf'
    figure = plt.figure(figsize=(8.27, 11.69))
    figure.text(0.1, 0.9, 'Controlled scientific article', fontsize=16)
    figure.text(0.1, 0.82, sample_text, fontsize=11, wrap=True)
    figure.savefig(pdf_path, format='pdf')
    plt.close(figure)
    pdf_bytes = pdf_path.read_bytes()

len(pdf_bytes), pdf_bytes[:4]

(11507, b'%PDF')

In [3]:
parser = LiteParseDocumentParser(max_pages=10, parse_timeout=20)
parsed = parser.parse_pdf(pdf_bytes)

assert parsed.parser_name == 'liteparse'
assert parsed.page_count == 1
assert 'Controlled scientific article' in parsed.text
assert len(parsed.text) >= 80

{
    'parser': parsed.parser_name,
    'pages': parsed.page_count,
    'characters': len(parsed.text),
    'preview': parsed.text[:300],
}

{'parser': 'liteparse',
 'pages': 1,
 'characters': 262,
 'preview': '# Controlled scientific article\n\nControlled scientific article. This document evaluates whether a treatment reduces symptoms in adults. The reported result is illustrative and must not be interpreted as clinical evidence. The purpose is validating local parsing.'}

## Resultado e limites

O teste confirma que PDFs podem ser processados localmente e entregues como texto rastreável ao restante do pipeline. Isso reduz o tamanho enviado ao Gemini e separa claramente parsing de interpretação.

O LiteParse não reconstrói de forma confiável os valores de gráficos e pode perder estrutura em tabelas densas, fórmulas ou digitalizações difíceis. Esses elementos devem gerar alerta ou uma rota de análise visual separada; nunca devem ser inferidos a partir de texto ausente. O próximo incremento do comportamento tipo NotebookLM é guardar vários documentos, fragmentá-los com página/origem e permitir perguntas respondidas somente com citações desses fragmentos.